In [ ]:
import pandas as pd

model_df = pd.read_csv(
    "../data/processed/leishmania_ml_dataset.csv"
)

model_df.head()

In [ ]:
from rdkit import Chem

model_df["Mol"] = model_df["SMILES"].apply(
    Chem.MolFromSmiles
)

print(model_df["Mol"].notnull().sum())

In [ ]:
from rdkit.Chem import AllChem

model_df["FP"] = model_df["Mol"].apply(
    lambda mol:
    AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=2,
        nBits=1024
    )
)

In [ ]:
subset = model_df.iloc[:500].copy()

In [ ]:
from rdkit.DataStructs import TanimotoSimilarity

activity_cliffs = []

for i in range(len(subset)):

    fp1 = subset.iloc[i]["FP"]
    p1 = subset.iloc[i]["pIC50"]

    for j in range(i+1, len(subset)):

        fp2 = subset.iloc[j]["FP"]
        p2 = subset.iloc[j]["pIC50"]

        similarity = TanimotoSimilarity(
            fp1,
            fp2
        )

        delta_activity = abs(p1 - p2)

        if (
            similarity >= 0.80
            and
            delta_activity >= 2
        ):

            activity_cliffs.append([
                subset.iloc[i]["Molecule ChEMBL ID"],
                subset.iloc[j]["Molecule ChEMBL ID"],
                similarity,
                delta_activity
            ])

In [ ]:
cliffs_df = pd.DataFrame(
    activity_cliffs,
    columns=[
        "Compound_1",
        "Compound_2",
        "Similarity",
        "Delta_pIC50"
    ]
)

cliffs_df.sort_values(
    "Delta_pIC50",
    ascending=False
).head(20)

In [ ]:
print(
    "Number of Activity Cliffs:",
    len(activity_cliffs)
)

In [ ]:
max_similarity = 0
max_delta = 0

for i in range(len(subset)):

    fp1 = subset.iloc[i]["FP"]
    p1 = subset.iloc[i]["pIC50"]

    for j in range(i + 1, len(subset)):

        fp2 = subset.iloc[j]["FP"]
        p2 = subset.iloc[j]["pIC50"]

        similarity = TanimotoSimilarity(fp1, fp2)
        delta = abs(p1 - p2)

        if similarity > max_similarity:
            max_similarity = similarity

        if delta > max_delta:
            max_delta = delta

print("Max Similarity:", round(max_similarity, 3))
print("Max Delta pIC50:", round(max_delta, 3))

In [ ]:
high_similarity_pairs = []

for i in range(len(subset)):

    fp1 = subset.iloc[i]["FP"]

    for j in range(i + 1, len(subset)):

        fp2 = subset.iloc[j]["FP"]

        similarity = TanimotoSimilarity(
            fp1,
            fp2
        )

        if similarity >= 0.80:

            delta = abs(
                subset.iloc[i]["pIC50"] -
                subset.iloc[j]["pIC50"]
            )

            high_similarity_pairs.append(
                [similarity, delta]
            )

print(
    "Number of pairs with similarity >= 0.80:",
    len(high_similarity_pairs)
)

In [ ]:
pd.DataFrame(
    high_similarity_pairs,
    columns=[
        "Similarity",
        "Delta_pIC50"
    ]
).sort_values(
    "Delta_pIC50",
    ascending=False
).head(20)

In [ ]:
activity_cliffs = []

for i in range(len(subset)):

    fp1 = subset.iloc[i]["FP"]
    p1 = subset.iloc[i]["pIC50"]

    for j in range(i + 1, len(subset)):

        fp2 = subset.iloc[j]["FP"]
        p2 = subset.iloc[j]["pIC50"]

        similarity = TanimotoSimilarity(
            fp1,
            fp2
        )

        delta_activity = abs(
            p1 - p2
        )

        if (
            similarity >= 0.80
            and
            delta_activity >= 1.0
        ):

            activity_cliffs.append([
                subset.iloc[i]["Molecule ChEMBL ID"],
                subset.iloc[j]["Molecule ChEMBL ID"],
                similarity,
                delta_activity,
                p1,
                p2
            ])

In [ ]:
cliffs_df = pd.DataFrame(
    activity_cliffs,
    columns=[
        "Compound_1",
        "Compound_2",
        "Similarity",
        "Delta_pIC50",
        "pIC50_1",
        "pIC50_2"
    ]
)

cliffs_df = cliffs_df.sort_values(
    "Delta_pIC50",
    ascending=False
)

print(
    "Number of Activity Cliffs:",
    len(cliffs_df)
)

cliffs_df.head(20)

## Activity Cliff Identification

Activity cliff analysis was performed using Morgan fingerprints (radius = 2, 1024 bits) and Tanimoto similarity. Compound pairs with structural similarity ≥ 0.80 and activity differences (ΔpIC50) ≥ 1.0 were considered activity cliffs. A total of 24 activity cliff pairs were identified among the first 500 compounds analyzed.

In [ ]:
top_cliffs = cliffs_df.head(10)

top_cliffs

In [ ]:
cliff_compounds = pd.unique(
    pd.concat([
        top_cliffs["Compound_1"],
        top_cliffs["Compound_2"]
    ])
)

cliff_structures = subset[
    subset["Molecule ChEMBL ID"].isin(
        cliff_compounds
    )
][
    [
        "Molecule ChEMBL ID",
        "SMILES",
        "pIC50"
    ]
]

cliff_structures

In [ ]:
cliff_structures.to_csv(
    "../results/top_activity_cliff_compounds.csv",
    index=False
)

In [ ]:
from rdkit.Chem import Draw

top_mols = [
    Chem.MolFromSmiles(smiles)
    for smiles in cliff_structures["SMILES"]
]

img = Draw.MolsToGridImage(
    top_mols[:12],
    molsPerRow=4,
    subImgSize=(300,300)
)

img

## Interpretation of Activity Cliffs

Several highly similar compound pairs exhibited substantial differences in biological activity despite sharing common structural scaffolds. The strongest activity cliffs displayed ΔpIC50 values greater than 1.0, corresponding to more than ten-fold differences in potency. These findings suggest that relatively small structural modifications can significantly influence anti-leishmanial activity and may provide valuable insights for lead optimization.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.hist(
    cliffs_df["Delta_pIC50"],
    bins=10
)

plt.xlabel("ΔpIC50")
plt.ylabel("Frequency")
plt.title("Distribution of Activity Cliffs")
plt.savefig(
    "../results/activity_cliff_distribution.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.scatter(
    cliffs_df["Similarity"],
    cliffs_df["Delta_pIC50"],
    alpha=0.7
)

plt.xlabel("Tanimoto Similarity")
plt.ylabel("ΔpIC50")

plt.title(
    "Activity Cliffs in Anti-Leishmanial Dataset"
)
plt.savefig(
    "../results/activity_cliff_scatter.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
cliffs_df.head(20).to_csv(
    "../results/top_20_activity_cliffs.csv",
    index=False
)

In [ ]:
cliffs_df.to_csv(
    "../results/activity_cliffs_full.csv",
    index=False
)